In [18]:
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

# Loading CSVs using path.cwd
PROJECT_ROOT = Path.cwd().parent
RAW_DATA = PROJECT_ROOT / "FIFA WC Dataset"
MODELS = PROJECT_ROOT / "models"
PROCESSED_DATA = PROJECT_ROOT / "processed_data"
ML_DATA = PROJECT_ROOT / "ml_data"
ML_DATA.mkdir(exist_ok=True)

model = joblib.load(MODELS / "lightgbm_tuned.pkl")
encoder = joblib.load(MODELS / "onehot_encoder.pkl")
feature_columns = joblib.load(MODELS / "feature_columns.pkl")


elo_df = pd.read_csv(PROCESSED_DATA / "elo_ratings_V3.csv")
team_matches = pd.read_csv(PROCESSED_DATA / "team_matches.csv")
train = pd.read_csv(ML_DATA / "train_V2.csv", nrows=0)



In [19]:
print("Model loaded successfully.")
print("Encoder loaded successfully.")
print(f"Number of model features: {len(feature_columns)}")

print("Elo Ratings Shape:", elo_df.shape)
print("Team Matches Shape:", team_matches.shape)

Model loaded successfully.
Encoder loaded successfully.
Number of model features: 127
Elo Ratings Shape: (210, 3)
Team Matches Shape: (47520, 8)


In [20]:
# ========== 1. User Inputs ==========

home_team = "Argentina"
away_team = "Portugal"
tournament = "FIFA World Cup"
neutral = 1

In [21]:
# 2. Helper Function - Get Team Elo Rating

def get_elo(team_name):
    
    team = elo_df.loc[elo_df["team"] == team_name]
    
    if team.empty:
        raise ValueError(f"Team '{team_name}' not found in Elo ratings.")
    
    return float(team.iloc[0]["elo_rating"])

In [22]:
print(f"{home_team}: {get_elo(home_team):.2f}")
print(f"{away_team}: {get_elo(away_team):.2f}")

Argentina: 2075.73
Portugal: 1916.75


In [23]:
# 3. Helper Function - Get Team Statistics


def get_team_stats(team_name):
    
    team_df = (
        team_matches[team_matches["team"] == team_name]
        .sort_values("date")
        .copy()
    )

    if team_df.empty:
        raise ValueError(f"No match history found for '{team_name}'.")

    # Overall statistics
    matches_played = len(team_df)
    wins = (team_df["result"] == "W").sum()

    win_pct = wins / matches_played

    avg_gf = team_df["goals_for"].mean()
    avg_ga = team_df["goals_against"].mean()

    # Last 5 matches
    # Last 5 matches
    last5 = team_df.tail(5)

    points = (
        (last5["result"] == "W").sum() * 3 +
        (last5["result"] == "D").sum()
    )

    last5_ppg = points / len(last5)

    last5_avg_gf = last5["goals_for"].mean()
    last5_avg_ga = last5["goals_against"].mean()

    return {
    "elo": float(get_elo(team_name)),
    "win_pct": float(win_pct),
    "avg_gf": float(avg_gf),
    "avg_ga": float(avg_ga),
    "last5_ppg": float(last5_ppg),
    "last5_avg_gf": float(last5_avg_gf),
    "last5_avg_ga": float(last5_avg_ga),
}

In [24]:
print(get_team_stats("Brazil"))
print(get_team_stats("Argentina"))

{'elo': 1940.57, 'win_pct': 0.6363636363636364, 'avg_gf': 2.0748663101604277, 'avg_ga': 0.7165775401069518, 'last5_ppg': 2.0, 'last5_avg_gf': 2.0, 'last5_avg_ga': 0.8}
{'elo': 2075.73, 'win_pct': 0.6326530612244898, 'avg_gf': 1.935860058309038, 'avg_ga': 0.8279883381924198, 'last5_ppg': 3.0, 'last5_avg_gf': 2.8, 'last5_avg_ga': 1.0}


In [25]:
# 4. Create Match Features

def create_match_features(home_team, away_team, neutral):
    """
    Creates the numerical feature vector for a match.
    """

    home = get_team_stats(home_team)
    away = get_team_stats(away_team)

    match_features = {
        "home_elo": home["elo"],
        "away_elo": away["elo"],
        "elo_diff": home["elo"] - away["elo"],

        "home_win_pct": home["win_pct"],
        "away_win_pct": away["win_pct"],

        "home_avg_gf": home["avg_gf"],
        "away_avg_gf": away["avg_gf"],

        "home_avg_ga": home["avg_ga"],
        "away_avg_ga": away["avg_ga"],

        "neutral": neutral,

        "home_last5_ppg": home["last5_ppg"],
        "away_last5_ppg": away["last5_ppg"],

        "home_last5_avg_gf": home["last5_avg_gf"],
        "away_last5_avg_gf": away["last5_avg_gf"],

        "home_last5_avg_ga": home["last5_avg_ga"],
        "away_last5_avg_ga": away["last5_avg_ga"],

        # Engineered Features
        "abs_elo_diff": abs(home["elo"] - away["elo"]),
        "win_pct_gap": home["win_pct"] - away["win_pct"],
        "last5_ppg_gap": home["last5_ppg"] - away["last5_ppg"],
        "avg_gf_gap": home["avg_gf"] - away["avg_gf"],
        "avg_ga_gap": home["avg_ga"] - away["avg_ga"],
    }

    return pd.DataFrame([match_features])

In [26]:
match_df = create_match_features(
    home_team,
    away_team,
    neutral
)

match_df


,home_elo,away_elo,elo_diff,home_win_pct,away_win_pct,home_avg_gf,away_avg_gf,home_avg_ga,away_avg_ga,neutral,...,away_last5_ppg,home_last5_avg_gf,away_last5_avg_gf,home_last5_avg_ga,away_last5_avg_ga,abs_elo_diff,win_pct_gap,last5_ppg_gap,avg_gf_gap,avg_ga_gap
0,2075.73,1916.75,158.98,0.632653,0.593472,1.93586,2.083086,0.827988,0.833828,1,...,1.6,2.8,1.6,1.0,0.6,158.98,0.039181,1.4,-0.147226,-0.00584


In [27]:
# 5. Encode Tournament

def encode_tournament(match_df, tournament):
    """
    One-hot encodes the tournament and appends it to the
    numerical match features.
    """

    # Tournament input
    tournament_input = pd.DataFrame({
        "tournament": [tournament]
    })

    # Encode tournament
    tournament_encoded = encoder.transform(tournament_input)

    # Convert to DataFrame
    tournament_df = pd.DataFrame(
        tournament_encoded,
        columns=encoder.get_feature_names_out(["tournament"])
    ).astype(int)

    # Combine
    match_df = pd.concat(
        [match_df.reset_index(drop=True),
         tournament_df.reset_index(drop=True)],
        axis=1
    )

    return match_df

In [28]:
match_df = encode_tournament(match_df, tournament)

print(match_df.shape)
match_df.head()

(1, 127)


,home_elo,away_elo,elo_diff,home_win_pct,away_win_pct,home_avg_gf,away_avg_gf,home_avg_ga,away_avg_ga,neutral,...,tournament_UEFA Euro,tournament_UEFA Euro qualification,tournament_UEFA Nations League,tournament_UNCAF Cup,tournament_USA Cup,tournament_United Arab Emirates Friendship Tournament,tournament_Unity Cup,tournament_VFF Cup,tournament_WAFF Championship,tournament_Windward Islands Tournament
0,2075.73,1916.75,158.98,0.632653,0.593472,1.93586,2.083086,0.827988,0.833828,1,...,0,0,0,0,0,0,0,0,0,0


In [29]:
match_df = match_df.reindex(columns=feature_columns, fill_value=0)
print(match_df.shape)
print(match_df.columns.equals(pd.Index(feature_columns)))

(1, 127)
True


In [30]:
# 6. Make Prediction

prediction = model.predict(match_df)[0]
probabilities = model.predict_proba(match_df)[0]

In [31]:
# Class Mapping

class_mapping = {
    0: "Away Win",
    1: "Draw",
    2: "Home Win"
}

predicted_result = class_mapping[prediction]

In [32]:
print("=" * 50)
print("        FIFA WORLD CUP PREDICTOR")
print("=" * 50)

print(f"Home Team : {home_team}")
print(f"Away Team : {away_team}")
print(f"Tournament: {tournament}")
print(f"Neutral   : {'Yes' if neutral else 'No'}")

print("\nPredicted Result")
print("-" * 20)
print(predicted_result)

print("\nPrediction Probabilities")
print("-" * 28)
print(f"Away Win : {probabilities[0]:.2%}")
print(f"Draw     : {probabilities[1]:.2%}")
print(f"Home Win : {probabilities[2]:.2%}")

        FIFA WORLD CUP PREDICTOR
Home Team : Argentina
Away Team : Portugal
Tournament: FIFA World Cup
Neutral   : Yes

Predicted Result
--------------------
Home Win

Prediction Probabilities
----------------------------
Away Win : 26.81%
Draw     : 20.48%
Home Win : 52.71%


In [33]:
# 7. Predict Match

def predict_match(home_team, away_team, tournament, neutral):
    """
    Predicts the outcome of a football match using the trained LightGBM model.
    """

    # Create numerical features
    match_df = create_match_features(
        home_team,
        away_team,
        neutral
    )

    # Encode tournament
    match_df = encode_tournament(
        match_df,
        tournament
    )

    # Align columns
    match_df = match_df.reindex(
        columns=feature_columns,
        fill_value=0
    )

    # Prediction
    prediction = model.predict(match_df)[0]
    probabilities = model.predict_proba(match_df)[0]

    class_mapping = {
        0: "Away Win",
        1: "Draw",
        2: "Home Win"
    }

    print("=" * 50)
    print("        FIFA WORLD CUP PREDICTOR")
    print("=" * 50)

    print(f"Home Team : {home_team}")
    print(f"Away Team : {away_team}")
    #print(f"Tournament: {tournament}")
    print(f"Neutral   : {'Yes' if neutral else 'No'}")

    print("\nPredicted Result")
    print("-" * 20)
    print(class_mapping[prediction])

    print("\nPrediction Probabilities")
    print("-" * 28)
    print(f"Away Win : {probabilities[0]:.2%}")
    print(f"Draw     : {probabilities[1]:.2%}")
    print(f"Home Win : {probabilities[2]:.2%}")

    return prediction, probabilities

In [34]:
predict_match(
    home_team,
    away_team,
    tournament,
    neutral
)

        FIFA WORLD CUP PREDICTOR
Home Team : Argentina
Away Team : Portugal
Neutral   : Yes

Predicted Result
--------------------
Home Win

Prediction Probabilities
----------------------------
Away Win : 26.81%
Draw     : 20.48%
Home Win : 52.71%


(np.int64(2), array([0.2681466 , 0.20478542, 0.52706798]))